# Revisión de la preparación de datos

Este notebook revisa los archivos generados por DVC. No crea ni modifica los datos oficiales.

## P1 — Tipado

**Pregunta:** ¿La primera transformación conserva todas las filas y convierte únicamente las fechas?

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data/raw/cfpb_reclamos_narrativa.parquet"
TYPED_PATH = PROJECT_ROOT / "data/interim/typed.parquet"

assert RAW_PATH.exists(), f"No se encontró {RAW_PATH}"
assert TYPED_PATH.exists(), f"No se encontró {TYPED_PATH}; ejecuta dvc repro"

### 1. Filas, columnas y tamaño

In [3]:
raw = pq.ParquetFile(RAW_PATH)
typed = pq.ParquetFile(TYPED_PATH)

file_summary = pd.DataFrame({
    "archivo": ["original", "tipado"],
    "filas": [raw.metadata.num_rows, typed.metadata.num_rows],
    "columnas": [raw.metadata.num_columns, typed.metadata.num_columns],
    "tamaño GiB": [RAW_PATH.stat().st_size / 1024**3, TYPED_PATH.stat().st_size / 1024**3],
})
file_summary["filas"] = file_summary["filas"].map("{:,}".format)
file_summary["tamaño GiB"] = file_summary["tamaño GiB"].map("{:.2f}".format)
display(file_summary)

assert typed.metadata.num_rows == raw.metadata.num_rows == 3_837_184
assert typed.metadata.num_columns == raw.metadata.num_columns == 16

,archivo,filas,columnas,tamaño GiB
0,original,"3,837,184",16,1.51
1,tipado,"3,837,184",16,0.85


### 2. Tipos antes y después

In [4]:
raw_schema = raw.schema_arrow
typed_schema = typed.schema_arrow
schema_comparison = pd.DataFrame({
    "columna": raw_schema.names,
    "tipo original": [str(field.type) for field in raw_schema],
    "tipo tipado": [str(field.type) for field in typed_schema],
})
schema_comparison["cambió"] = schema_comparison["tipo original"] != schema_comparison["tipo tipado"]
display(schema_comparison)

changed_columns = schema_comparison.loc[schema_comparison["cambió"], "columna"].tolist()
assert changed_columns == ["Date received", "Date sent to company"]
assert typed_schema.field("Date received").type == pa.date32()
assert typed_schema.field("Date sent to company").type == pa.date32()

,columna,tipo original,tipo tipado,cambió
0,Date received,large_string,date32[day],True
1,Product,large_string,large_string,False
2,Sub-product,large_string,large_string,False
3,Issue,large_string,large_string,False
4,Sub-issue,large_string,large_string,False
5,Consumer complaint narrative,large_string,large_string,False
6,Company public response,large_string,large_string,False
7,Company,large_string,large_string,False
8,State,large_string,large_string,False
9,ZIP code,large_string,large_string,False


### 3. Calidad de las fechas

In [5]:
typed_dates = pq.read_table(TYPED_PATH, columns=["Date received", "Date sent to company"])
date_summary = []
for column in typed_dates.column_names:
    values = typed_dates[column]
    date_summary.append({
        "columna": column,
        "mínimo": pc.min(values).as_py(),
        "máximo": pc.max(values).as_py(),
        "nulos": values.null_count,
    })
display(pd.DataFrame(date_summary))
assert all(row["nulos"] == 0 for row in date_summary)

,columna,mínimo,máximo,nulos
0,Date received,2015-03-19,2026-07-27,0
1,Date sent to company,2015-03-19,2026-08-13,0


### 4. Identificadores

In [6]:
complaint_ids = pq.read_table(TYPED_PATH, columns=["Complaint ID"])["Complaint ID"]
unique_ids = pc.count_distinct(complaint_ids).as_py()
display(pd.DataFrame({
    "medida": ["filas", "IDs únicos", "IDs nulos"],
    "valor": [len(complaint_ids), unique_ids, complaint_ids.null_count],
}))
assert unique_ids == len(complaint_ids) == 3_837_184
assert complaint_ids.null_count == 0

,medida,valor
0,filas,3837184
1,IDs únicos,3837184
2,IDs nulos,0


## Resultado de P1

- Se conservan las 3,837,184 filas y las 16 columnas.
- Solo cambian `Date received` y `Date sent to company`: pasan de texto a fecha.
- No se crean fechas nulas.
- Los 3,837,184 identificadores continúan presentes y son únicos.
- Ninguna categoría, narrativa u objetivo se modifica en esta etapa.